In [1]:
"""
Build the BLIMMP-Explorer node and adjacency JSONs.

Both node frequencies and edge weights now come from the AllTheBacteria
domain-level priors (21,401 assemblies), so the two channels agree.
"""
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper")

# Regenerate these graphs AFTER the ProgramNode union fix.
KEGG_GRAPHS = BASE / "KEGG_Graph_Generation" / "KEGG_Graphs_Generated_13AUG2026"

KO_FREQ_FILE = (BASE / "Bayesian Priors: ATB Analysis" / "ATB_Taxonomy" /
                "ATB Frequency" / "ko_freq_ko_matrix_sampleids_domain_level_priors.tsv")

EDGE_PRIORS_FILE = (BASE / "KEGG_Neighbors_Generation" / "MODULE_ALL_NEIGHBOR_DATA" /
                    "Module_AllHop_Refilled_domain_level_priors.json")

OUT_NODES = Path("all_module_nodes_13AUG2026.json")
OUT_LINKS = Path("all_module_adjacency_links_13AUG2026.json")

for p in (KEGG_GRAPHS, KO_FREQ_FILE, EDGE_PRIORS_FILE):
    if not p.exists():
        raise FileNotFoundError(f"missing input: {p}")


def load_ko_frequencies(path: Path) -> dict:
    """KOs \t KO_count \t KO_frequency  (frequency is precomputed, do not divide)."""
    freq = {}
    with path.open() as f:
        header = next(f).rstrip("\n").split("\t")
        if header[:1] != ["KOs"]:
            raise ValueError(f"unexpected header: {header}")
        for lineno, line in enumerate(f, start=2):
            line = line.rstrip("\n")
            if not line.strip():
                continue
            fields = line.split("\t")
            if len(fields) != 3:
                raise ValueError(f"{path.name} line {lineno}: expected 3 fields, got {len(fields)}")
            ko, _count, frequency = fields
            freq[ko] = float(frequency)
    return freq


def load_edge_priors(path: Path) -> dict:
    """
    {KO_a: {"_count": n_genomes_with_a, KO_b: cooccurrence_count, ...}}
    Returns {(a, b): P(b present | a present)}.
    """
    with path.open() as f:
        raw = json.load(f)
    edges = {}
    for src, row in raw.items():
        total = row.get("_count", 0)
        if not total:
            continue
        for tgt, count in row.items():
            if tgt == "_count":
                continue
            edges[(src, tgt)] = count / total
    return edges


def strip_suffix(node: str) -> str:
    """K00844_1 -> K00844.  START/SINK pass through unchanged."""
    return node.split("_")[0]


def process_nodes(graph_dir: Path, ko_freq: dict) -> dict:
    result = {}
    unmatched = set()
    for path in sorted(graph_dir.glob("module_*_nodes.json")):
        module_id = path.stem.split("_")[1]
        with path.open() as f:
            nodes = json.load(f)
        enriched = []
        for node, group in nodes.items():
            ko = strip_suffix(node)
            if ko in ("START", "SINK"):
                occ = 0.0
            else:
                if ko not in ko_freq:
                    unmatched.add(ko)
                occ = round(ko_freq.get(ko, 0.0), 5)
            enriched.append({"id": node, "group": group, "KO_Occurrence": occ})
        result[module_id] = sorted(enriched, key=lambda x: x["group"])
    if unmatched:
        print(f"  note: {len(unmatched)} KOs absent from frequency table, set to 0")
    return result


def process_adjacency(graph_dir: Path, edges: dict) -> dict:
    result = {}
    for path in sorted(graph_dir.glob("module_*_adjacency.json")):
        module_id = path.stem.split("_")[1]
        with path.open() as f:
            adj = json.load(f)
        links = []
        for src, targets in adj.items():
            src_ko = strip_suffix(src)
            for tgt in targets:
                tgt_ko = strip_suffix(tgt)
                prob = edges.get((src_ko, tgt_ko), 0.0)
                links.append({"source": src, "target": tgt,
                              "edge_occurence": round(prob, 5)})
        result[module_id] = {"links": links}
    return result


def report(nodes: dict, links: dict) -> None:
    occ = [n["KO_Occurrence"] for m in nodes for n in nodes[m]]
    over = sum(1 for x in occ if x > 1.0)
    print(f"\nmodules: {len(nodes)} nodes / {len(links)} adjacency")
    print(f"KO_Occurrence  range [{min(occ):.5f}, {max(occ):.5f}]  above 1.0: {over}")
    if over:
        print("  WARNING: frequencies above 1.0 indicate a normalization problem")

    real = [L for m in links for L in links[m]["links"]
            if strip_suffix(L["source"]) not in ("START", "SINK")
            and strip_suffix(L["target"]) not in ("START", "SINK")]
    if real:
        eo = [L["edge_occurence"] for L in real]
        zero = sum(1 for x in eo if x == 0)
        print(f"edge_occurence range [{min(eo):.5f}, {max(eo):.5f}]")
        print(f"KO->KO links: {len(real)}  zero-weight: {zero} ({100*zero/len(real):.1f}%)")


if __name__ == "__main__":
    ko_freq = load_ko_frequencies(KO_FREQ_FILE)
    print(f"loaded {len(ko_freq)} KO frequencies")

    edges = load_edge_priors(EDGE_PRIORS_FILE)
    print(f"loaded {len(edges)} directed KO pairs")

    nodes = process_nodes(KEGG_GRAPHS, ko_freq)
    links = process_adjacency(KEGG_GRAPHS, edges)

    with OUT_NODES.open("w") as f:
        json.dump(nodes, f, indent=2)
    with OUT_LINKS.open("w") as f:
        json.dump(links, f, indent=2)

    report(nodes, links)
    print(f"\nwrote {OUT_NODES}\nwrote {OUT_LINKS}")

loaded 25829 KO frequencies
loaded 26750 directed KO pairs
  note: 28 KOs absent from frequency table, set to 0

modules: 340 nodes / 340 adjacency
KO_Occurrence  range [0.00000, 0.99687]  above 1.0: 0
edge_occurence range [0.00000, 1.00000]
KO->KO links: 3710  zero-weight: 539 (14.5%)

wrote all_module_nodes_13AUG2026.json
wrote all_module_adjacency_links_13AUG2026.json
